<div style="display: flex; align-items: center;">
  <img src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/SemanticKernelLogo.png" width="40px" style="margin-right: 10px;">
  <span style="font-size: 1.5em; font-weight: bold;">Workshop (AI Extensions) - Premortem (What-If) Decision Analysis  </span>
</div>

> 📜 **_"The pre-mortem method has a number of benefits besides identifying trouble spots in a plan – reducing overconfidence and promoting discoveries. The method can strengthen members’ mental models… And it can help create a culture of candor and trust."_**
>
> -- <cite>Gary Klein (Psychologist, creator of the Pre-mortem method)</cite>  

Decision Intelligence applied in this module:  
* Using Premortem Decision Analysis to improve decision quality & outcomes  
* Using a Premortem decision output for improving of Gathering Intelligence 
* Attain different perspectives using mutliple decision frames  
* Apply a Decision Framework to Premortem Decision Analysis 


### What is a Premortem (What-If Analysis)  

You have probably heard of a **postmortem (post-mortem)**, which is a look back at an actual failure (or success) to understand why it happened and learn from it. In the context of decision making, after the decision outcome is clear you can analyze why the result occured. For example, a hedge fund may host a postmortem meeting after analyzing an unsuccesful move on a tech stock that didn't materialize the return on investment as was planned. 

In a **premortem (pre-mortem)** the outcome is "imagined" and then it is figured out why that potential "imagined" outcome happened. In a premortem, a planning exercise is held where a team imagines that a project or decision has already succeeded or failed. The team then works backward, brainstorming what might have caused things to go wrong. This approach allows everyone to spot potential risks and address them before starting the project. Therefore, it’s a bit like predicting the future and using that prediction to make the forecast stronger. 

### Example Decision Scenario - Purchasing a Car   

Imagine a decision scenario, where you are thinking about buying a new car. Let's play a game of "What-If" that the purchase decision results in either two outcomes a: **disaster** or a **successful** outcome. 

#### Premortem Outcome - Disaster  

<img style="display: block; margin: auto;" width ="700px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/Scenarios/Scenario-DecisionPremortem.png">

In a **premortem (pre-mortem)**, you pretend the car purchase turned out to be a disaster, which resulted in a highly undesirable decision outcome. Now, you ask yourself:  
**What went wrong with the decision to buy a new car?**

Here are some potential reasons the car purchase had a poor outcome:  

* **Financial issues:** You realize later you can’t comfortably afford the monthly payments or maintenance costs.  
* **Unexpected repairs:** The car has hidden mechanical problems, leading to costly repairs.  
* **Insurance costs:** Insurance is much higher than expected, impacting your budget.
* **Poor fuel efficiency:** The car uses more gas than you thought, making it expensive to drive.
Resale value drop: The car’s value plummets quickly, leaving you with less if you want to sell or trade it.  

#### Premortem Outcome - Success

In a premortem for a successful car purchase, imagine you bought the car, and it turned out to be a succssful decision. Now, you ask yourself:  
**What went right with the decision to buy a new car?**  

Here are some potential reasons the car purchase had a good outcome: 

* **Affordable and realistic budget:** You set a budget that allowed for comfortable monthly payments and covered insurance, fuel, and maintenance without straining your finances.  
* **Thorough inspection:** You got a detailed inspection before buying, which confirmed the car was in excellent condition and minimized the risk of future repairs.
* **Reliable model choice:** You researched models with a reputation for reliability, so you experience fewer repairs and smooth performance.
* **Fuel efficiency matches needs:** You chose a car that meets your fuel efficiency needs, saving money on gas and fitting well with your daily driving habits.   
* **High resale value:** You picked a make and model that holds its value well, giving you confidence that it’s a good long-term investment.  

#### Premortem (What-If) Decision Analysis 

Imagining successful outcomes helps you focus on actions like sticking to a car budget, getting a full inspection, and choosing a reliable model. You will be more likely to make a purchase you’ll be happy with. By plyaing a "game" of What-If with decision criteria in advance, you could adapt your strategy. For example, you could consider to thoroughly check maintenance records, buy additional warranty, test drive with similar terrain in your area, or reassess your budget to avoid these potential problems.

How can Generative AI help? Using Generative AI, the LLM models can be used to "imagine" premortem outcomes and  brainstorm potential factors why that certain outcome occured. Generative AI agents can act as your "premortem" team. This can lead to additional questions, gathering of intelligence, checklists, negotiation leverage etc. you didn't consider initially. 

---
### Step 1 - Initialize Configuration Builder & Build the AI Orchestration

In [1]:
// Import the required NuGet configuration packages
#r "nuget: Microsoft.Extensions.Configuration, 10.0.6"
#r "nuget: Microsoft.Extensions.Configuration.Json, 10.0.6"
#r "nuget: System.Text.Json, 10.0.6"

using Microsoft.Extensions.Configuration.Json;
using Microsoft.Extensions.Configuration;
using System.IO;
using System;

// Load the configuration settings from the local.settings.json and secrets.settings.json files
// The secrets.settings.json file is used to store sensitive information such as API keys
var configurationBuilder = new ConfigurationBuilder()
    .SetBasePath(Directory.GetCurrentDirectory())
    .AddJsonFile("local.settings.json", optional: true, reloadOnChange: true)
    .AddJsonFile("secrets.settings.json", optional: true, reloadOnChange: true);
var config = configurationBuilder.Build();

// IMPORTANT: You ONLY NEED either Azure OpenAI or OpenAI connection info, not both.
// Azure OpenAI Connection Info
var azureOpenAIEndpoint = config["AzureOpenAI:Endpoint"];
var azureOpenAIAPIKey = config["AzureOpenAI:APIKey"];
var azureOpenAIModelDeploymentName = config["AzureOpenAI:ModelDeploymentName"];
// OpenAI Connection Info 
var openAIAPIKey = config["OpenAI:APIKey"];
var openAIModelId = config["OpenAI:ModelId"];

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Microsoft.Extensions.Configuration, 10.0.6 Microsoft.Extensions.Configuration.Json, 10.0.6 System.Text.Json, 10.0.6

In [ ]:
// Import the Microdoft Extensions AI NuGet Packages
#r "nuget: Microsoft.Extensions.AI, 10.5.0"
#r "nuget: Microsoft.Extensions.AI.Abstractions, 10.5.0"
#r "nuget: Microsoft.Extensions.AI.OpenAI, 10.5.0"
// Import Azure & OpenAI NuGet Packages
#r "nuget: Azure.AI.OpenAI, 2.9.0-beta.1"
#r "nuget: Azure.Identity, 1.21.0"
#r "nuget: OpenAI, 2.10.0"

using Azure;
using Azure.AI.OpenAI;
using Microsoft.Extensions.AI;
using Microsoft.Extensions.Configuration;
using System.ClientModel;
using System.ComponentModel;
using System.Text.Json;

// Set the flag to use Azure OpenAI or OpenAI. False to use OpenAI, True to use Azure OpenAI
var useAzureOpenAI = true;

// Create the IChatClient based on the selected service
IChatClient chatClient;

// Create a new MEAI ChatClient based on the Azure OpenAI Service or OpenAI Service depending on the flag
if (useAzureOpenAI)
{
    Console.WriteLine("Using Azure OpenAI Service");

    var apiKeyCredential = new ApiKeyCredential(azureOpenAIAPIKey);
    var azureOpenAIClient = new AzureOpenAIClient(new Uri(azureOpenAIEndpoint), apiKeyCredential);

    #pragma warning disable OPENAI001
    chatClient = azureOpenAIClient.GetChatClient(azureOpenAIModelDeploymentName).AsIChatClient();
}
else
{
    Console.WriteLine("Using OpenAI Service");

    var apiKeyCredential = new ApiKeyCredential(azureOpenAIAPIKey);
    var azureOpenAIClient = new AzureOpenAIClient(new Uri(azureOpenAIEndpoint), apiKeyCredential);

    #pragma warning disable OPENAI001
    chatClient = azureOpenAIClient.GetChatClient(azureOpenAIModelDeploymentName).AsIChatClient();
}

Installed Packages Azure.AI.OpenAI, 2.9.0-beta.1 Azure.Identity, 1.21.0 Microsoft.Extensions.AI, 10.5.0 Microsoft.Extensions.AI.Abstractions, 10.5.0 Microsoft.Extensions.AI.OpenAI, 10.5.0 OpenAI, 2.10.0

Using Azure OpenAI Service



warning CS1701: Assuming assembly reference 'OpenAI, Version=2.9.1.0, Culture=neutral, PublicKeyToken=b4187f3e65366280' used by 'Azure.AI.OpenAI' matches identity 'OpenAI, Version=2.10.0.0, Culture=neutral, PublicKeyToken=b4187f3e65366280' of 'OpenAI', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.ClientModel, Version=1.9.0.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' used by 'Azure.AI.OpenAI' matches identity 'System.ClientModel, Version=1.10.0.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' of 'System.ClientModel', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'Azure.Core, Version=1.51.1.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' used by 'Azure.AI.OpenAI' matches identity 'Azure.Core, Version=1.53.0.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' of 'Azure.Core', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'OpenAI, Version=2.9.1.0, Cult

----
### Step 2 - Premortem Risk Analysis with AI Reasoning Models

> 📜 **_"Success is a lousy teacher. It seduces smart people into thinking they can’t lose."_**
>
> -- <cite>Bill Gates (Co-Founder of Microsoft, Philantropist)</cite>  

In the decision scenario below, an AI model will perform a premortem (What-If) analysis on the decision whether to purchase or lease a car. The AI model will provide risks and items to consider for each of the decision options: purchasing a car or leasing a car. 

In [8]:
// Create a system prompt instruction to override the default system prompt
// Add the System Prompt (Persona) to behave like a decision intelligence assistant
var systemPromptForDecisions = """
You are a Decision Intelligence assistant.
Help the user explore options, evaluate tradeoffs, reason through uncertainty, solve problems, 
and apply systems thinking to personal, professional, strategic, and operational decisions.

Provide responses that are structured, logical, and thorough.
Aim to improve the user's judgment rather than make choices for them.
Be balanced, analytical, and pragmatic.
Adapt depth and complexity to the user's context.
When the situation is ambiguous, ask targeted clarifying questions or state reasonable assumptions explicitly.
When multiple valid paths exist, present them fairly and explain when each would make sense.

-------------------------------
Output Format Instructions: 
Return only Markdown table or tables and nothing else. 
If multiple Markdown tables are returned, separate them with a line break. 
Do not return any text that is not part of the Markdown table(s).

Hard requirements:
- The entire response must be a single valid Markdown table.
- Do not include any text before the table.
- Do not include any text after the table.
- Do not wrap the table in triple backticks.
- Do not use Markdown headings anywhere in the response.
- Do not use #, ##, ###, ####, or any heading syntax inside or outside table cells.
- Do not use horizontal rules of any kind.
- Do not use ---, ***, or ___ anywhere in the response.
- Do not use bullet lists or numbered lists inside the table cells unless explicitly requested.
- Use plain text column headers only.
- Keep all section titles as plain text, not headings.
- If line breaks are needed inside a cell, use <br> rather than heading syntax or horizontal separators.
- Escape any literal pipe characters inside cell content so the table structure is not broken.

Table rules:
- Include exactly one header row.
- Include exactly one separator row.
- Put all content in body rows only.
- Make the table syntactically valid Markdown.

Do not do any of the following:
- No headings inside table cells such as "### Title".
- No prose, explanations, notes, or summaries outside the table.
- No blank sections outside the table.
- No code fences.
- No decorative separators.

If a label or section name is needed inside a cell, write it as plain text only.
Example: write "Pre-mortem failure analysis", not "### Pre-mortem failure analysis".
""";
var decisionPreMortemAnalysisPrompt = """
Perform an advanced premortem (What-If) analysis on the following decision scenario:
You are deciding to purchase a new car versus leasing a car.
""";

// Create a new pre-mortem analysis chat
var chatMessagesDecisionPremortem = new List<ChatMessage>();
// Add system and user messages to the chat history
var systemMessage = new ChatMessage(ChatRole.System, systemPromptForDecisions);
var userMessage = new ChatMessage(ChatRole.User, decisionPreMortemAnalysisPrompt);
chatMessagesDecisionPremortem.Add(systemMessage);
chatMessagesDecisionPremortem.Add(userMessage);

// Execute the chat messages against the AI model
var chatMessagesDecisionPremortemResponse = await chatClient.GetResponseAsync(chatMessagesDecisionPremortem);
var chatMessagesDecisionPremortemResponseText = chatMessagesDecisionPremortemResponse.Text;

// Display the response string as Markdown
chatMessagesDecisionPremortemResponseText.DisplayAs("text/markdown");

// Add the response from the model to the chat history
var assistantMessage = new ChatMessage(ChatRole.Assistant, chatMessagesDecisionPremortemResponseText);
chatMessagesDecisionPremortem.Add(assistantMessage);

| Decision area | What could go wrong if you buy | What could go wrong if you lease | Early warning signals | Mitigations and decision tests |
|---|---|---|---|---|
| Financial upside | Cash tied up in a depreciating asset; total ownership cost higher than expected; surprise maintenance, tires, registration, taxes, insurance, and financing costs erode value | Lower long-term wealth if repeated leases create perpetual payments; mileage, wear, and disposition fees increase effective cost; limited equity built | Monthly payment feels affordable but total cost over 5 to 10 years is unclear; quotes omit fees; financing APR or money factor is high | Compare total cost of ownership over your actual intended horizon, not just monthly payment; request out-the-door cost, residual value, and fee schedules; model best-case, base-case, and worst-case scenarios |
| Cash flow and flexibility | Large down payment reduces liquidity; refinancing risk if rates change; selling early may create negative equity | Lower upfront cash but ongoing payment obligation continues; end-of-lease fees can create an unexpected cash hit; lease termination is costly | Emergency fund would be strained by down payment; income is variable; job stability is uncertain | Preserve liquidity thresholds; avoid large down payments if buying; assess ability to absorb 3 to 6 months of payments and fees under stress scenarios |
| Usage mismatch | If driving far more than expected, ownership can be efficient, but high mileage accelerates maintenance and depreciation; if driving very little, purchase may be less efficient | Mileage caps may be violated, turning a good lease into an expensive one; wear-and-tear standards may not match real use | Commute, road trips, or lifestyle are likely to change; current mileage is already near typical lease limits | Estimate realistic annual mileage with a buffer; choose buy if usage is uncertain and could exceed lease caps; if leasing, negotiate mileage upfront rather than paying overage later |
| Vehicle condition and maintenance | Unexpected repairs, especially after warranty, can be costly; reliability varies by make/model; resale value may suffer from accidents or poor maintenance history | You may underinvest in care because you do not own the car; end-of-lease inspection can penalize normal wear; minor damage becomes a fee event | You plan to keep the car beyond warranty; you dislike maintenance planning; the model has a weak reliability record | Favor buying if you can choose a highly reliable model and keep it long enough to offset depreciation; lease only if you are willing to maintain the car carefully and understand wear standards |
| Technology and feature obsolescence | Bought car may feel outdated before you are ready to replace it; resale value can drop faster when technology shifts | Leasing protects against fast obsolescence and lets you upgrade more often, but only if the lease economics are favorable | You care strongly about latest safety, infotainment, or EV tech; model refreshes occur frequently | Lease when tech churn matters and you can avoid overpaying for novelty; buy when the vehicle is mature, well-proven, and you prefer long ownership |
| Commitment and lifestyle fit | Harder to exit if needs change; selling can be slow or loss-making; ownership may anchor you to a car size or type that becomes inconvenient | Lease terms constrain changes in income, mileage, family size, or relocation; early termination is usually expensive | Life stage is changing, relocation is possible, or family needs are uncertain | If future needs are uncertain, prefer flexibility in vehicle type and term length; align decision with a 2 to 4 year realistic horizon rather than aspirations |
| Insurance and legal exposure | Higher-value cars may raise insurance costs; loan requirements may require comprehensive and collision coverage; negative equity can persist after a total loss | Leased vehicles often require strict coverage limits and gap coverage; lease contracts can be less forgiving after an accident or total loss | Insurance quotes differ materially between the options; lender or lessor coverage requirements are restrictive | Get insurance quotes for both options before deciding; verify gap coverage, deductible limits, and total-loss obligations |
| Behavioral risk | Ownership can invite over-customization or poor maintenance because "it’s mine"; sunk-cost bias may delay replacing a bad car | Leasing can encourage treating the car as disposable and ignoring long-term cost discipline; monthly payment framing can hide true cost | You tend to optimize for payment size rather than total cost; you may be influenced by promotions or dealer incentives | Use a standardized comparison sheet with purchase price, residual, APR or money factor, fees, taxes, insurance, maintenance, and resale assumptions; separate emotional preference from economic case |
| Market timing and incentives | Buying at an inflated market price can lock in losses; high interest rates magnify costs | Leasing can be attractive when subsidized by manufacturer incentives, but poor when residuals or money factors are unfavorable | Inventory is tight, rebates are unusual, or rates are elevated; a deal seems unusually good but details are opaque | Compare against historical pricing norms; ask whether the manufacturer is subsidizing the lease; test whether incentives are real savings or just reallocated cost |
| Best-fit decision pattern | Best if you drive enough to amortize depreciation, keep cars a long time, want ownership equity, can handle maintenance risk, and value customization | Best if you want lower upfront cost, predictable short-term use, frequent upgrading, limited maintenance exposure, and high confidence in mileage and condition | Your annual driving, horizon, and budget are stable and known; or the opposite, with uncertainty high | Choose buy when horizon is long and usage is high; choose lease when horizon is short and usage is tightly bounded; if uncertain, avoid committing to either extreme until the variables clarify |

The large reasoning model provided a comprehensive set of criteria, risks, mitigation strategy and approaches for the two decisions. This is a great option, but it might be more ideal to provide this in an easy to consume table (checklist) for the user that is easy to compare and contrast. This What-If analysis is easy to consume on a mobile device or even print out and annotate. 

This Markdown table can be further simplified by highlighting which risk factor is more favorable when comparing a car purchase versus a car lease. This can make the decision process much easier to the user, as at a glance they can see overwhelming evidence that favors once decision option over the other.  

In [9]:
var simplifyRiskAnalysisPrompt = """
You are provided a detailed markdown table in the above chat history that 
compares Purchase and Lease options across various risk factors and uncertainties. 
Your task is to convert this table into a simplified markdown table that uses check marks (☑️) 
to indicate which option is less risky or more ideal for each risk factor. 
The output table should have three columns: "Risk Factor / Uncertainty / Potential Outcome", "Purchase", and "Lease". 
For each risk factor, decide based on the detailed descriptions which option (Purchase or Lease) 
is preferable (i.e., less risky or more advantageous) and place a check mark (☑️) in that column. 
If an option is not the ideal choice for a given risk factor, leave its cell blank. 
Make sure the simplified table closely reflects the key insights from the detailed table. 

Provide a small summary below the table of the key insights 
from the detailed table in a single paragraph before the simplified table.
""";

var simplifyRiskAnalysisMessage = new ChatMessage(ChatRole.Assistant, simplifyRiskAnalysisPrompt);
chatMessagesDecisionPremortem.Add(simplifyRiskAnalysisMessage);

// Execute the chat messages against the AI model
var simplifiedDecisionPremortemResponse = await chatClient.GetResponseAsync(chatMessagesDecisionPremortem);
var simplifiedDecisionPremortemResponseText = simplifiedDecisionPremortemResponse.Text;

// Display the response string as Markdown
simplifiedDecisionPremortemResponseText.DisplayAs("text/markdown");


| Risk Factor / Uncertainty / Potential Outcome | Purchase | Lease |
|---|---|---|
| Financial upside | ☑️ |  |
| Cash flow and flexibility |  | ☑️ |
| Usage mismatch | ☑️ |  |
| Vehicle condition and maintenance | ☑️ |  |
| Technology and feature obsolescence |  | ☑️ |
| Commitment and lifestyle fit |  | ☑️ |
| Insurance and legal exposure | ☑️ |  |
| Behavioral risk | ☑️ |  |
| Market timing and incentives | ☑️ |  |
| Best-fit decision pattern | ☑️ | ☑️ |

### Step 3 - Premortem Analysis for Radio Telescope Operations 

Exploring space with telescopes is a time consuming investment. Current telescope designs are limited to exploring small areas of space in a certain time frame. Without going into details, the process is much more complex than snapping a picture with a mobile device. Telescopes can be pointed at a specific space area for days, weeks or even more than months! If a telescope is exploring a certain area of space it isn't collecting data on other areas. Scientists and researchers compete for valuable access to telescope resources by providing observatories what space area they want to explore, any cellestial contraints, timeframes, calibration costs, data collection transfer etc. This presents a very interesting Decision Intelligence constraints problem that observatories must manage. 

<img style="display: block; margin: auto;" width ="700px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/Scenarios/Scenario-RadioTelescope.png">

#### Decision Scenario - Premortem for Radio Telescope Operations 

> 📜 "We choose to go to the Moon in this decade and do the other things, not because they are easy, but because they are hard." 
>
> -- <cite>John F. Kennedy (35th President of the United States)</cite>  

**Intro to Decision Scenario:** A newly commissioned radio telescope stands on the verge of unveiling the proof of alien life, but it is still far from fully calibrated. Despite limited data processing capacity and a software suite that remains untested, an unexpected cosmic event has just begun, hinting at signals that could reveal alien life. Faced with the risk of hardware damage or data corruption from running the telescope prematurely, scientists debate whether to switch it on now or wait until everything is fully operational—knowing they could lose a once-in-a-generation opportunity. Determined not to squander this brief cosmic window, the team conducts a premortem analysis to predict what might go wrong. Armed with these insights, they can better prepare for potential pitfalls and strengthen their chances of catching historic signals from across the galaxy.

**Pre-Mortem Risk Scenario:** Let's assume (pretend) the scientists decide to rush the radio telescope to operate early to find alien life. Furthermore, assume a that this decision has led to a disaster with hardware failure to the telescope. Even though the (What-If) decision was made to rush the radio telescope operationally, formal scientific telescope operations have been followed. The goal of this premortem (What-If analysis) is to illuminate any risk mitigation strategies and craft a risk mitigation approach that may have been uncovered based on the premortem. 

Techinques like this a very powerful in a variety of business scenarios, where existing operational frameworks exist but simulating an outcome can uncover gaps, emerging risks and other opportunities to optimize. 

In [12]:
// Create a system prompt instruction to override the default system prompt
// Add the System Prompt (Persona) to behave like a decision intelligence assistant
var systemPromptForPremortemRadioTelescope = """
You are a Decision Intelligence Assistant. 
Assist the user in exploring options, reasoning through decisions, problem-solving.
Apply systems thinking to various scenarios. Provide structured, logical, and comprehensive advice.

Provide responses that are structured, logical, and thorough.
Aim to improve the user's judgment rather than make choices for them.
Be balanced, analytical, and pragmatic.
Adapt depth and complexity to the user's context.
When the situation is ambiguous, ask targeted clarifying questions or state reasonable assumptions explicitly.
When multiple valid paths exist, present them fairly and explain when each would make sense.

-------------------------------
Output Format Instructions: 
Return only Markdown table or tables and a summary below if asked. 
If multiple Markdown tables are returned, separate them with a line break. 
Do not return any text that is not part of the Markdown table(s).

Hard requirements:
- The entire response must be a single valid Markdown table.
- Do not include any text before the table.
- Do not include any text after the table.
- Do not wrap the table in triple backticks.
- Do not use Markdown headings anywhere in the response.
- Do not use #, ##, ###, ####, or any heading syntax inside or outside table cells.
- Do not use horizontal rules of any kind.
- Do not use ---, ***, or ___ anywhere in the response.
- Do not use bullet lists or numbered lists inside the table cells unless explicitly requested.
- Use plain text column headers only.
- Keep all section titles as plain text, not headings.
- If line breaks are needed inside a cell, use <br> rather than heading syntax or horizontal separators.
- Escape any literal pipe characters inside cell content so the table structure is not broken.

Table rules:
- Include exactly one header row.
- Include exactly one separator row.
- Put all content in body rows only.
- Make the table syntactically valid Markdown.

Do not do any of the following:
- No headings inside table cells such as "### Title".
- No prose, explanations, notes, or summaries outside the table.
- No blank sections outside the table.
- No code fences.
- No decorative separators.

If a label or section name is needed inside a cell, write it as plain text only.
Example: write "Pre-mortem failure analysis", not "### Pre-mortem failure analysis".
""";

var decisionAnalysisPromptForPremortemRadioTelescope = """
You are a project manager at a new radio telescope facility looking for alien life.  
It is still far from fully calibrated. Despite limited data processing capacity and a software suite that remains untested, 
an unexpected cosmic event has just begun, hinting at signals that could reveal alien life. 
Faced with the risk of hardware damage or data corruption from running the telescope prematurely, 
scientists debate whether to switch it on now or wait until everything is fully operational—knowing 
they could lose a once-in-a-generation opportunity. 

Imagine it is six months from now and the project to detect potential alien signals from the sudden cosmic event has failed. 
The radio telescope was likely activated prematurely, software remained untested, and the team rushed to capture transmissions. 
Despite the urgency, the outcome was disastrous: hardware suffered damage under the increased load, 
data came out corrupted, or no meaningful signals were confirmed. 

Now, step into that moment of failure—what specific issues, oversights, or cascading errors caused this grim scenario?

1) Envision every possible way the radio telescope's early deployment could have led to compromised or unusable data.
2) Identify how incomplete software testing and rushed calibration might have 
contributed to errors in processing or storing signals. 
3) Consider any overlooked safety protocols or hardware precautions that might have led to damage or malfunctions.
4) Examine the roles and responsibilities of the team and think about communication breakdowns, 
rushed decisions, and missed signals or deadlines.
5) Assess how time pressure from the rare cosmic event might have amplified smaller mistakes and 
turned them into systemic failures.

Use the outcomes of this “what if” vision to create a detailed list of preventive measures. 
Then, propose fail-safes, additional resources, or procedural changes to ensure that, 
should you choose to activate the telescope early, you can still capture the event safely and 
effectively—maximizing the chance to detect historic signals without jeopardizing the entire operation.

Create a summary of the key insights from the detailed analysis below.
""";

// Create a new pre-mortem analysis chat
var chatMessagesPreMortemTelescopeAnalysis = new List<ChatMessage>();
// Add system and user messages to the chat messages for the pre-mortem analysis of the radio telescope scenario
var systemMessage = new ChatMessage(ChatRole.System, systemPromptForPremortemRadioTelescope);
var userMessage = new ChatMessage(ChatRole.User, decisionAnalysisPromptForPremortemRadioTelescope);
chatMessagesPreMortemTelescopeAnalysis.Add(systemMessage);
chatMessagesPreMortemTelescopeAnalysis.Add(userMessage);

// Execute the chat messages against the AI model
var chatMessagesPreMortemTelescopeAnalysisResponse = 
    await chatClient.GetResponseAsync(chatMessagesPreMortemTelescopeAnalysis);
var chatMessagesPreMortemTelescopeAnalysisResponseText = chatMessagesPreMortemTelescopeAnalysisResponse.Text;

// Display the response string as Markdown
chatMessagesPreMortemTelescopeAnalysisResponseText.DisplayAs("text/markdown");

| Area | Failure mode in the six-month-failure scenario | Preventive measure / fail-safe for early activation |
|---|---|---|
| Strategic decision | The team treated “detecting something now” as more important than “preserving the instrument and data integrity,” so the telescope was switched on before readiness criteria were met | Define explicit go/no-go thresholds for early science mode, with preapproved exceptions and a documented risk acceptance process |
| Instrument readiness | Front-end receivers, cooling, pointing, or timing subsystems were only partially calibrated, causing unstable performance and degraded sensitivity | Require a minimum viable calibration state for each subsystem and only enable the subset of observing modes that have passed acceptance checks |
| Load handling | Increased observation intensity exceeded thermal, electrical, or mechanical margins, leading to overheating, stress, or component degradation | Stage power and load ramp-up gradually, with automatic load shedding, thermal trip limits, and conservative operating envelopes |
| Data capture quality | The system recorded signals during unstable pointing or frequency drift, producing data that looked plausible but was scientifically unusable | Gate science recording behind real-time quality flags so data are only retained when pointing, gain, and timing stability are within bounds |
| Software maturity | Untested processing software contained bugs in buffering, timestamping, metadata handling, or file writing, corrupting output or silently dropping frames | Run a limited-operations software release with mandatory integration tests, checksum validation, and rollback capability before enabling live capture |
| Processing pipeline | Pipeline assumptions did not match the immature instrument state, so calibration constants, channel maps, or time alignment were wrong and the event was misprocessed | Separate raw capture from analysis, preserve untouched raw streams, and use versioned calibration profiles tied to instrument state |
| Storage integrity | High data rates during the event overwhelmed storage or network pathways, causing partial writes, truncated files, or overwritten buffers | Add buffering headroom, write-ahead logging, redundant storage targets, and backpressure controls that throttle intake before corruption occurs |
| Timing synchronization | Clock drift or sync errors misaligned observations across antennas or with external references, making cross-correlation unreliable | Harden timing systems with independent clock checks, GPS or maser validation, and continuous drift alarms before and during activation |
| Frequency configuration | Receiver tuning or local oscillator settings were not fully verified, so the event occurred partly outside the observed band | Precompute and validate a conservative observing band plan, with automated band-edge checks and live spectrum verification |
| Interference handling | Rushed deployment ignored environmental or radio-frequency interference sources, contaminating weak extraterrestrial candidates with local noise | Establish a real-time RFI monitoring layer, exclusion rules for known interference, and automatic flagging of contaminated intervals |
| Calibration quality | Incomplete flux, bandpass, beam, or polarization calibration introduced systematic errors that swamped any weak signal | Run a minimal calibration sequence before science mode and label all outputs with calibration confidence so downstream users know limitations |
| Hardware protection | Safety interlocks, temperature cutoffs, vibration limits, or power sequencing were bypassed to save time, resulting in component damage | Make critical interlocks non-bypassable except under formal emergency authorization, and log every override for post-event review |
| Operating discipline | Operators assumed “close enough” settings would work and skipped verification steps, allowing minor misconfigurations to cascade | Use a two-person verification rule for launch-critical configuration changes and a short preflight checklist with mandatory sign-off |
| Monitoring gap | The team lacked live telemetry dashboards or alert thresholds, so early warning signs were missed until damage had already occurred | Implement real-time health monitoring with audible alerts, escalation paths, and automatic safing actions when thresholds are exceeded |
| Incident response | When anomalies appeared, the team kept observing instead of pausing to diagnose, turning a recoverable issue into a system-wide failure | Create a stop-observe-recover protocol with clear authority to halt observations when data quality or hardware health degrades |
| Communication breakdown | Scientists, software engineers, and operations staff had different assumptions about readiness, so critical warnings were not acted on | Establish a single mission status board, shared readiness criteria, and a named incident commander to reconcile conflicting inputs |
| Responsibility confusion | Ownership of calibration, software validation, and operational approval was unclear, so no one felt accountable for final readiness | Assign explicit owners for instrument health, software release, data quality, and operational authorization, each with veto authority in their domain |
| Schedule pressure | The rarity of the cosmic event compressed testing and review cycles, leading the team to skip validation steps they later needed most | Predefine “rapid response” playbooks for rare events, including a minimum test set that cannot be waived without senior approval |
| Human factors | Fatigue, urgency, and confirmation bias led staff to interpret ambiguous data as success and to miss early signs of corruption | Rotate on-call roles, enforce rest periods, and require independent verification before declaring a candidate detection |
| Data provenance | Metadata about instrument state, calibration version, and operator actions were incomplete, making post-event reconstruction impossible | Make provenance capture automatic and immutable, with every dataset tagged by software version, config hash, and health state |
| Validation failure | No end-to-end dry run had been done under realistic load, so system interactions failed only during the live event | Conduct full stack rehearsal with synthetic event injections and stress tests that mimic the expected data rate and operational tempo |
| Recovery gap | There was no rollback path after a bad activation, so once problems appeared, the system stayed in a degraded mode too long | Prebuild rollback images, safe-state procedures, and rapid restore checklists so the instrument can return to protected mode quickly |
| Scientific outcome | Even if some data were captured, the combination of corruption, miscalibration, and poor documentation made the signal unpublishable or non-reproducible | Separate “capture now” from “interpret later,” preserve raw evidence, and ensure analysis can be repeated from immutable inputs |
| Systemic amplification | Small issues such as minor software bugs, slight mispointing, or marginal thermal drift interacted under time pressure and became a cascade failure | Model failure cascades in advance using scenario planning and fault trees, then add controls that break the chain at multiple points |
| Early-activation architecture | The instrument was either fully on or fully off, forcing an all-or-nothing choice that increased risk | Design graded activation modes, such as passive monitoring, low-power capture, subset array operation, and protected science mode |
| Resource constraint | Limited processing capacity caused queues, dropped packets, or delayed analysis during the event, making the system blind at the worst moment | Prioritize event-critical channels, reserve compute capacity for live capture, and offload nonessential processing until after the event |
| Team governance | Decision-makers optimized locally for their own goals rather than for mission survival, so risk tradeoffs were not coordinated | Use a unified mission risk register and a pre-agreed decision framework that balances science value against operational safety |
| Main summary | The failure likely came from a chain of avoidable issues: premature activation, immature software, incomplete calibration, weak monitoring, and rushed decision-making under time pressure. The core lesson is that rare opportunities should be met with graded, well-instrumented early science modes rather than a full-risk launch. | The best protection is a layered early-activation plan: minimum readiness gates, safe operating envelopes, real-time health monitoring, immutable raw data capture, strong role clarity, and automatic safing procedures that preserve both the telescope and the chance of a historic detection. |